### mmvec Heatmap with MASST, directionality, and ClassyFire superclass annotations

In [4]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from os.path import join

In [5]:
# Data 
cond = pd.read_csv(
    "../data/mmvec/annotated_conditionals_SZ_MC_PD_0507202_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
)

original_ids = pd.read_csv(
    "../data/mmvec/subset_conditionals_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
).index.astype(str)

info_df = pd.read_csv(
    os.path.expanduser(
        "../data/mmvec/info_feature_complete_Britta_Shiseido_FBMN_2_attempt3quant_SZ_MC_PD_05072025.csv"
    )
)

VIPs_extreme_Load = pd.read_csv(
    "../data/mmvec/VIPs_extreme_Load_data_SZ_MC_PD_05072025_cytoscapeannotation.csv"
)

masst = pd.read_csv(
    "../data/mmvec/masst_screening_results.csv"
)

# Assign colors
superclass_colors = {
    "Alkaloids and derivatives": "#e41a1c", "Benzenoids": "#377eb8",
    "Hydrocarbons": "#4daf4a", "Lignans, neolignans and related compounds": "#984ea3",
    "Lipids and lipid-like molecules": "#ff7f00", "Nucleosides, nucleotides, and analogues": "#a65628",
    "Organic 1,3-dipolar compounds": "#f781bf", "Organic acids and derivatives": "#2ca25e",
    "Organic nitrogen compounds": "#66c2a5", "Organic oxygen compounds": "#8da0cb",
    "Organic salts": "#fc8d62", "Organoheterocyclic compounds": "#e78ac3",
    "Organosulfur compounds": "#a6d854", "Phenylpropanoids and polyketides": "#ffd92f",
    "Unknown": "#bdbdbd"
}
groupcontrib_colors = {"Low": "#4575b4", "High": "#e08214"}

masst_categories = [
    "plants", "tissue", "food", "microbes", "microbiome", "personal care product"
]
masst_colors = {
    "plants": "#458550", "tissue": "#393a92", "food": "#3391c1",
    "microbes": "#76325f", "microbiome": "#f08976", "personal care product": "#e8bad8"
}

# Feature mappings
feature_to_superclass = {
    str(int(r['Feature'])): (r['ClassyFire#superclass'] if pd.notna(r['ClassyFire#superclass']) else "Unknown")
    for _, r in info_df.iterrows() if not pd.isna(r['Feature'])
}

groupcontrib_dict = dict(zip(
    VIPs_extreme_Load['ID'].astype(str),
    VIPs_extreme_Load['GroupContrib']
))

masst_dict = (
    masst[masst['Matched_Size'] > 0]
    .groupby('Compound_ID')['Category']
    .apply(list)
    .to_dict()
)

# Color Bars
col_super = []
col_sebum = []
col_masst = {cat: [] for cat in masst_categories}

for fid in original_ids:
    fid_str = str(fid)
    col_super.append(superclass_colors.get(
        feature_to_superclass.get(fid_str, "Unknown"), "#bdbdbd"))
    col_sebum.append(groupcontrib_colors.get(
        groupcontrib_dict.get(fid_str, None), "#ffffff"))
    hits = masst_dict.get(int(fid), [])
    for cat in masst_categories:
        col_masst[cat].append(masst_colors[cat] if cat in hits else "#ffffff")

all_col_colors = [col_super, col_sebum] + [col_masst[cat] for cat in masst_categories]

# Center data
heatmap_data = cond.T
heatmap_data_centered = heatmap_data - heatmap_data.mean()

# Compound name remapping
compound_name_map = {
    "Monoelaidin": "Glycerol-C18:1",
    "Glycerol 1-myristate": "Glycerol-C14:0",
    "Monopalmitolein (9c)": "Glycerol-C16:1",
    "Alanine/Sarcosine-C15:0": "Ala/Sar-C15:0",
    # add more as needed
}

# Apply renaming
heatmap_data_centered = heatmap_data_centered.rename(index=compound_name_map)

# Plot
sns.set(font_scale=0.6)
g = sns.clustermap(
    heatmap_data_centered,
    cmap="RdBu_r",
    col_colors=all_col_colors,
    method="average",
    metric="euclidean",
    figsize=(15, 12),
    dendrogram_ratio=(.1, .02),
    cbar_pos=(0.02, .8, .03, .18)
)
# Colorbar label
g.ax_cbar.set_ylabel("Log Conditional Probability", fontsize=22, rotation=90, labelpad=10)
g.ax_cbar.yaxis.set_label_position("left")
g.ax_cbar.tick_params(axis='y', which='both', labeltop=True, labelbottom=False)
g.ax_cbar.tick_params(labelsize=16)  # or try 18/20 to match the rest of the figure

# Feature name (tick) font sizes for metabolites (y) and microbes (x)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=22, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=22)


g.fig.suptitle("Top Co-occurring Untargeted Metabolites and Microbial Genera Based on Conditional Probability", fontsize=24, y=1.04)


# Label Annotation Bars to the Right of the Color Bars
labels = ["Superclass", "Sebum group"] + [f"MASST: {cat}" for cat in masst_categories]

# Add labels using ax_col_colors
for i, label in enumerate(labels):
    g.ax_col_colors.text(
        1.01,                             # a bit to the right
        1 - (i + 0.5) / len(labels),      # evenly space vertically
        label,
        transform=g.ax_col_colors.transAxes,
        fontsize=22,
        ha='left', va='center'
    )

# Structured Legend 
handles = []
# Superclass block
handles.append(mpatches.Patch(color='none', label='ClassyFire Superclass'))
handles += [mpatches.Patch(color=c, label=l) for l, c in superclass_colors.items()]
# Sebum block
handles.append(mpatches.Patch(color='none', label='Sebum group'))
handles += [
    mpatches.Patch(color=groupcontrib_colors["Low"], label="Low Sebum (< 3.5 $\mu$g/cm²)"),
    mpatches.Patch(color=groupcontrib_colors["High"], label="High Sebum (> 16.9 $\mu$g/cm²)")
]
# MASST block
handles.append(mpatches.Patch(color='none', label='MASST categories'))
handles += [mpatches.Patch(color=c, label=l) for l, c in masst_colors.items()]

g.fig.legend(
    handles=handles,
    loc='upper left',
    bbox_to_anchor=(g.ax_heatmap.get_position().x1 + 0.3, g.ax_heatmap.get_position().y1+0.7),
    borderaxespad=0.,
    fontsize=22,
    frameon=False
)


# Save outputs 
outdir = "../figures/main"
os.makedirs(outdir, exist_ok=True)

for ext in ("pdf", "png", "svg"):
    plt.savefig(join(outdir, f"Figure6b_t2.{ext}"), bbox_inches="tight")

plt.show()
print("Saved heatmap with 8 stacked annotation bars.")


Saved heatmap with 8 stacked annotation bars.


/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_47128/1061508218.py:170: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [13]:
import re
import os
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from os.path import join

# Data 
cond = pd.read_csv(
    "../data/mmvec/annotated_conditionals_SZ_MC_PD_0507202_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
)

original_ids = pd.read_csv(
    "../data/mmvec/subset_conditionals_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
).index.astype(str)

info_df = pd.read_csv(
    os.path.expanduser(
        "../data/mmvec/info_feature_complete_Britta_Shiseido_FBMN_2_attempt3quant_SZ_MC_PD_05072025.csv"
    )
)

VIPs_extreme_Load = pd.read_csv(
    "../data/mmvec/VIPs_extreme_Load_data_SZ_MC_PD_05072025_cytoscapeannotation.csv"
)

masst = pd.read_csv(
    "../data/mmvec/masst_screening_results.csv"
)

# Assign colors
superclass_colors = {
    "Alkaloids and derivatives": "#e41a1c", "Benzenoids": "#377eb8",
    "Hydrocarbons": "#4daf4a", "Lignans, neolignans and related compounds": "#984ea3",
    "Lipids and lipid-like molecules": "#ff7f00", "Nucleosides, nucleotides, and analogues": "#a65628",
    "Organic 1,3-dipolar compounds": "#f781bf", "Organic acids and derivatives": "#2ca25e",
    "Organic nitrogen compounds": "#66c2a5", "Organic oxygen compounds": "#8da0cb",
    "Organic salts": "#fc8d62", "Organoheterocyclic compounds": "#e78ac3",
    "Organosulfur compounds": "#a6d854", "Phenylpropanoids and polyketides": "#ffd92f",
    "Unknown": "#bdbdbd"
}
groupcontrib_colors = {"Low": "#4575b4", "High": "#e08214"}

masst_categories = [
    "plants", "tissue", "food", "microbes", "microbiome", "personal care product"
]
masst_colors = {
    "plants": "#458550", "tissue": "#393a92", "food": "#3391c1",
    "microbes": "#76325f", "microbiome": "#f08976", "personal care product": "#e8bad8"
}

# Feature mappings
feature_to_superclass = {
    str(int(r['Feature'])): (r['ClassyFire#superclass'] if pd.notna(r['ClassyFire#superclass']) else "Unknown")
    for _, r in info_df.iterrows() if not pd.isna(r['Feature'])
}

groupcontrib_dict = dict(zip(
    VIPs_extreme_Load['ID'].astype(str),
    VIPs_extreme_Load['GroupContrib']
))

masst_dict = (
    masst[masst['Matched_Size'] > 0]
    .groupby('Compound_ID')['Category']
    .apply(list)
    .to_dict()
)

# Color Bars
col_super = []
col_sebum = []
col_masst = {cat: [] for cat in masst_categories}

for fid in original_ids:
    fid_str = str(fid)
    col_super.append(superclass_colors.get(
        feature_to_superclass.get(fid_str, "Unknown"), "#bdbdbd"))
    col_sebum.append(groupcontrib_colors.get(
        groupcontrib_dict.get(fid_str, None), "#ffffff"))
    hits = masst_dict.get(int(fid), [])
    for cat in masst_categories:
        col_masst[cat].append(masst_colors[cat] if cat in hits else "#ffffff")

all_col_colors = [col_super, col_sebum] + [col_masst[cat] for cat in masst_categories]

# Center data
heatmap_data = cond.T
heatmap_data_centered = heatmap_data - heatmap_data.mean()

# Custom name overrides (applied before plotting)
compound_name_map = {
    "Monoelaidin": "Glycerol-C18:1",
    "Glycerol 1-myristate": "Glycerol-C14:0",
    "Monopalmitolein (9c)": "Glycerol-C16:1",
    "Alanine/Sarcosine-C15:0": "Ala/Sar-C15:0",
}

# Build feature ID → compound name mapping
feature_to_name = {
    str(int(r['Feature'])): (r['Compound_Name'] if pd.notna(r['Compound_Name']) else str(int(r['Feature'])))
    for _, r in info_df.iterrows() if not pd.isna(r['Feature'])
}

# Rename columns using feature_to_name
heatmap_data_centered.columns = [
    feature_to_name.get(str(col), str(col))
    for col in heatmap_data_centered.columns
]

# Apply custom overrides to column names
heatmap_data_centered = heatmap_data_centered.rename(columns=compound_name_map)


def clean_label(label):
    # First apply compound name overrides
    for old, new in compound_name_map.items():
        if old in label:
            label = label.replace(old, new)
    # Strip (Feature XXXX)
    label = re.sub(r'\s*\(Feature \d+\)', '', label)
    # Reformat standalone "mz:X, RT:Y" → "(X, Y)" to match annotated style
    label = re.sub(r'mz:([\d.]+),\s*RT:([\d.]+)', r'(\1, \2)', label)
    return label.strip()    

# Plot
sns.set(font_scale=0.6)
g = sns.clustermap(
    heatmap_data_centered,
    cmap="RdBu_r",
    col_colors=all_col_colors,
    method="average",
    metric="euclidean",
    figsize=(15, 12),
    dendrogram_ratio=(.1, .02),
    cbar_pos=(0.02, .8, .03, .18)
)

# Colorbar label
g.ax_cbar.set_ylabel("Log Conditional Probability", fontsize=22, rotation=90, labelpad=10)
g.ax_cbar.yaxis.set_label_position("left")
g.ax_cbar.tick_params(axis='y', which='both', labeltop=True, labelbottom=False)
g.ax_cbar.tick_params(labelsize=16)

# Apply cleaned tick labels
new_xlabels = [clean_label(t.get_text()) for t in g.ax_heatmap.get_xticklabels()]
g.ax_heatmap.set_xticklabels(new_xlabels, fontsize=22, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=22)

g.fig.suptitle("Top Co-occurring Untargeted Metabolites and Microbial Genera Based on Conditional Probability", fontsize=24, y=1.04)

# Label Annotation Bars
labels = ["Superclass", "Sebum group"] + [f"MASST: {cat}" for cat in masst_categories]
for i, label in enumerate(labels):
    g.ax_col_colors.text(
        1.01,
        1 - (i + 0.5) / len(labels),
        label,
        transform=g.ax_col_colors.transAxes,
        fontsize=22,
        ha='left', va='center'
    )

# Structured Legend
handles = []
handles.append(mpatches.Patch(color='none', label='ClassyFire Superclass'))
handles += [mpatches.Patch(color=c, label=l) for l, c in superclass_colors.items()]
handles.append(mpatches.Patch(color='none', label='Sebum group'))
handles += [
    mpatches.Patch(color=groupcontrib_colors["Low"], label="Low Sebum (< 3.5 $\mu$g/cm²)"),
    mpatches.Patch(color=groupcontrib_colors["High"], label="High Sebum (> 16.9 $\mu$g/cm²)")
]
handles.append(mpatches.Patch(color='none', label='MASST categories'))
handles += [mpatches.Patch(color=c, label=l) for l, c in masst_colors.items()]

g.fig.legend(
    handles=handles,
    loc='upper left',
    bbox_to_anchor=(g.ax_heatmap.get_position().x1 + 0.3, g.ax_heatmap.get_position().y1 + 0.7),
    borderaxespad=0.,
    fontsize=22,
    frameon=False
)

# Save outputs
outdir = "../figures/main"
os.makedirs(outdir, exist_ok=True)

for ext in ("pdf", "png", "svg"):
    plt.savefig(join(outdir, f"Figure6b_t2.{ext}"), bbox_inches="tight")

plt.show()
print("Saved heatmap with 8 stacked annotation bars.")

Saved heatmap with 8 stacked annotation bars.


/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_47128/942227839.py:195: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [18]:
import re
import os
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from os.path import join

# ============================================================
# Data 
# ============================================================
cond = pd.read_csv(
    "../data/mmvec/annotated_conditionals_SZ_MC_PD_0507202_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
)

original_ids = pd.read_csv(
    "../data/mmvec/subset_conditionals_topASV40Met_PLSDA_06022025.tsv",
    sep='\t', index_col=0
).index.astype(str)

info_df = pd.read_csv(
    os.path.expanduser(
        "../data/mmvec/info_feature_complete_Britta_Shiseido_FBMN_2_attempt3quant_SZ_MC_PD_05072025.csv"
    )
)

VIPs_extreme_Load = pd.read_csv(
    "../data/mmvec/VIPs_extreme_Load_data_SZ_MC_PD_05072025_cytoscapeannotation.csv"
)

masst = pd.read_csv(
    "../data/mmvec/masst_screening_results.csv"
)

# ============================================================
# Assign colors
# ============================================================
superclass_colors = {
    "Alkaloids and derivatives": "#e41a1c", "Benzenoids": "#377eb8",
    "Hydrocarbons": "#4daf4a", "Lignans, neolignans and related compounds": "#984ea3",
    "Lipids and lipid-like molecules": "#ff7f00", "Nucleosides, nucleotides, and analogues": "#a65628",
    "Organic 1,3-dipolar compounds": "#f781bf", "Organic acids and derivatives": "#2ca25e",
    "Organic nitrogen compounds": "#66c2a5", "Organic oxygen compounds": "#8da0cb",
    "Organic salts": "#fc8d62", "Organoheterocyclic compounds": "#e78ac3",
    "Organosulfur compounds": "#a6d854", "Phenylpropanoids and polyketides": "#ffd92f",
    "Unknown": "#bdbdbd"
}
groupcontrib_colors = {"Low": "#4575b4", "High": "#e08214"}

masst_categories = [
    "plants", "tissue", "food", "microbes", "personal care product"
]
masst_colors = {
    "plants": "#458550", "tissue": "#393a92", "food": "#3391c1",
    "microbes": "#76325f", "microbiome": "#f08976", "personal care product": "#e8bad8"
}

# Feature mappings
feature_to_superclass = {
    str(int(r['Feature'])): (r['ClassyFire#superclass'] if pd.notna(r['ClassyFire#superclass']) else "Unknown")
    for _, r in info_df.iterrows() if not pd.isna(r['Feature'])
}

groupcontrib_dict = dict(zip(
    VIPs_extreme_Load['ID'].astype(str),
    VIPs_extreme_Load['GroupContrib']
))

masst_dict = (
    masst[masst['Matched_Size'] > 0]
    .groupby('Compound_ID')['Category']
    .apply(list)
    .to_dict()
)

# Feature -> (m/z, RT) mapping
MZ_COL = "mz"
RT_COL = "RT"
feature_to_mzrt = {
    str(int(r['Feature'])): (r[MZ_COL], r[RT_COL])
    for _, r in info_df.iterrows()
    if not pd.isna(r['Feature'])
}

# Color Bars
col_super = []
col_sebum = []
col_masst = {cat: [] for cat in masst_categories}

for fid in original_ids:
    fid_str = str(fid)
    col_super.append(superclass_colors.get(
        feature_to_superclass.get(fid_str, "Unknown"), "#bdbdbd"))
    col_sebum.append(groupcontrib_colors.get(
        groupcontrib_dict.get(fid_str, None), "#ffffff"))
    hits = masst_dict.get(int(fid), [])
    for cat in masst_categories:
        col_masst[cat].append(masst_colors[cat] if cat in hits else "#ffffff")

all_col_colors = [col_sebum, col_super] + [col_masst[cat] for cat in masst_categories]

# Center data
heatmap_data = cond.T
heatmap_data_centered = heatmap_data - heatmap_data.mean()

# Custom name overrides (applied before plotting)
compound_name_map = {
    "Monoelaidin": "Glycerol-C18:1",
    "Glycerol 1-myristate": "Glycerol-C14:0",
    "Monopalmitolein (9c)": "Glycerol-C16:1",
    "Alanine/Sarcosine-C15:0": "Ala/Sar-C15:0",
}

feature_to_name = {
    str(int(r['Feature'])): (r['Compound_Name'] if pd.notna(r['Compound_Name']) else str(int(r['Feature'])))
    for _, r in info_df.iterrows() if not pd.isna(r['Feature'])
}

heatmap_data_centered.columns = [
    feature_to_name.get(str(col), str(col))
    for col in heatmap_data_centered.columns
]

heatmap_data_centered = heatmap_data_centered.rename(columns=compound_name_map)

original_feature_order = list(original_ids)


def clean_label(label, feature_id=None):
    # Apply compound name overrides
    for old, new in compound_name_map.items():
        if old in label:
            label = label.replace(old, new)

    # Strip "(Feature XXXX)"
    label = re.sub(r'\s*\(Feature \d+\)', '', label)

    # ------------------------------------------------------------
    # Strip ANY pre-existing mz/RT text anywhere in the label,
    # in either style: "mz:123.45, RT:4.32" or "m/z 123.45, RT 4.32"
    # whether or not it's wrapped in parentheses.
    # ------------------------------------------------------------
    label = re.sub(
        r'\(?\s*m/?z\s*:?\s*[\d.]+\s*,\s*RT\s*:?\s*[\d.]+\s*\)?',
        '',
        label,
        flags=re.IGNORECASE
    )

    # Strip any now-empty or leftover trailing parenthetical (e.g. "()" or stray "(  )")
    label = re.sub(r'\s*\(\s*\)\s*$', '', label)
    # Strip any other trailing parenthetical too (covers "(123.45, 4.32)" style leftovers)
    label = re.sub(r'\s*\([^()]*\)\s*$', '', label).strip()

    # Rebuild the bracket with the correct m/z + RT for this feature
    if feature_id is not None:
        mzrt = feature_to_mzrt.get(str(feature_id))
        if mzrt is not None:
            mz, rt = mzrt
            try:
                bracket = f"(m/z {float(mz):.2f}, RT {float(rt):.2f})"
            except (TypeError, ValueError):
                bracket = f"(m/z {mz}, RT {rt})"
            label = f"{label} {bracket}"

    return label.strip()


# Plot
sns.set(font_scale=0.6)
g = sns.clustermap(
    heatmap_data_centered,
    cmap="RdBu_r",
    col_colors=all_col_colors,
    method="average",
    metric="euclidean",
    figsize=(15, 12),
    dendrogram_ratio=(.1, .02),
    cbar_pos=(0.02, .8, .03, .18)
)

g.ax_cbar.set_ylabel("Log Conditional Probability", fontsize=22, rotation=90, labelpad=10)
g.ax_cbar.yaxis.set_label_position("left")
g.ax_cbar.tick_params(axis='y', which='both', labeltop=True, labelbottom=False)
g.ax_cbar.tick_params(labelsize=16)

col_order = g.dendrogram_col.reordered_ind
new_xlabels = []
for idx in col_order:
    raw_label = heatmap_data_centered.columns[idx]
    feat_id = original_feature_order[idx]
    new_xlabels.append(clean_label(raw_label, feature_id=feat_id))

g.ax_heatmap.set_xticklabels(new_xlabels, fontsize=22, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=22)

g.fig.suptitle("Top Co-occurring Untargeted Metabolites and Microbial Genera Based on Conditional Probability", fontsize=24, y=1.04)

labels = ["Sebum group", "Superclass"] + [f"MASST: {cat}" for cat in masst_categories]
for i, label in enumerate(labels):
    g.ax_col_colors.text(
        1.01,
        1 - (i + 0.5) / len(labels),
        label,
        transform=g.ax_col_colors.transAxes,
        fontsize=22,
        ha='left', va='center'
    )

handles = []
handles.append(mpatches.Patch(color='none', label='Sebum group'))
handles += [
    mpatches.Patch(color=groupcontrib_colors["Low"], label="Low Sebum (< 3.5 $\mu$g/cm²)"),
    mpatches.Patch(color=groupcontrib_colors["High"], label="High Sebum (> 16.9 $\mu$g/cm²)")
]
handles.append(mpatches.Patch(color='none', label='ClassyFire Superclass'))
handles += [mpatches.Patch(color=c, label=l) for l, c in superclass_colors.items()]
handles.append(mpatches.Patch(color='none', label='MASST categories'))
handles += [mpatches.Patch(color=masst_colors[c], label=c) for c in masst_categories]

g.fig.legend(
    handles=handles,
    loc='upper left',
    bbox_to_anchor=(g.ax_heatmap.get_position().x1 + 0.3, g.ax_heatmap.get_position().y1 + 0.7),
    borderaxespad=0.,
    fontsize=22,
    frameon=False
)

outdir = "../figures/main"
os.makedirs(outdir, exist_ok=True)

for ext in ("pdf", "png", "svg"):
    plt.savefig(join(outdir, f"Figure6b_t2.{ext}"), bbox_inches="tight")

plt.show()
print("Saved heatmap with 7 stacked annotation bars (sebum first, microbiome MASST removed, mz/RT bracket replaced).")

Saved heatmap with 7 stacked annotation bars (sebum first, microbiome MASST removed, mz/RT bracket replaced).


/var/folders/v_/f2630xts0zgcr95tld13kd4c0000gn/T/ipykernel_47128/1541235975.py:237: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
